**Case 1: Bhotekoshi landslide, Nepal (August 2026)** <br/>

On 25 August 2026, part of a glacier on Langtang Lirung broke off and fell into the valley below, damming the river. The next morning the dam burst, and a flood of water and debris swept down the Bhote Koshi, through Rasuwagadhi, Timure and Syabrubesi, killing more than 1,400 people

This notebook looks at what NISAR saw in the weeks before, during and after: whether anything changed beforehand that could have been an early sign, what the flood changed along the valley, and what stood in its way

**Main sources** <br/>

- [Challenge resources](https://www.spaceappschallenge.org/2026/challenges/dancing-with-the-sars/?tab=resources)

- [NASA Earthdata Search](https://search.earthdata.nasa.gov/search)

- [CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html)

- [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/)

- [Earthdata Login](https://urs.earthdata.nasa.gov/)

**Summary** <br/>

- 
- 
- 
- 


<br/>
<br/>

### 1. Data Retrieval (API Connection)

Find the NISAR data for this case through **[CMR (Common Metadata Repository)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr)**, NASA's catalogue of every Earthdata dataset ([Earthdata Search](https://www.earthdata.nasa.gov/learn/earthdata-search) runs on it), then check if we can read the actual files

- Searching CMR is public, no login needed ([CMR search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html))

- The data files need an Earthdata login ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview))

#### 1.1 NISAR collections

- **Level 1 (in radar coordinates)**: radar geometry, not on a map yet

- **Level 2 (in geographic coordinates)**: put on a map grid and terrain corrected, with a quality mask

- **Level 3 (derived geophysical parameters)**: a physical quantity worked out from Level 2, e.g. soil moisture

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview))

<br/>

Each dataset in CMR is a **collection** with a short name. A keyword search for "NISAR" lists them all ([keyword search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#c-keyword)), then keep the Level 2 and Level 3 ones

In [11]:
# view NISAR collections
import requests
import pandas as pd
pd.set_option("display.max_colwidth", 80)

CMR = "https://cmr.earthdata.nasa.gov/search"

resp = requests.get(f"{CMR}/collections.json", params={"keyword": "NISAR", "page_size": 100}, timeout=60)
entries = resp.json()["feed"]["entry"]

collections = pd.DataFrame([(e["short_name"], e["title"], e["time_start"][:10]) for e in entries],
                           columns=["short_name", "title", "data_starts"])

# keep the level 2 and 3 science products, plus urgent response
collections = collections[collections.short_name.str.match(r"NISAR_(L2|L3|UR_L2)")].sort_values("short_name")
collections

,short_name,title,data_starts
8,NISAR_L2_GCOV_BETA_V1,NISAR Beta Geocoded Polarimetric Covariance Product (Version 1),2025-10-01
0,NISAR_L2_GCOV_PROVISIONAL_V1,NISAR Provisional Geocoded Polarimetric Covariance Product (Version 1),2025-10-28
22,NISAR_L2_GCOV_V1,NISAR Validated Geocoded Polarimetric Covariance Product (Version 1),2025-10-28
14,NISAR_L2_GOFF_BETA_V1,NISAR Beta Geocoded Pixel Offsets (Version 1),2025-10-01
5,NISAR_L2_GOFF_PROVISIONAL_V1,NISAR Provisional Geocoded Pixel Offsets (Version 1),2025-10-28
23,NISAR_L2_GOFF_V1,NISAR Validated Geocoded Pixel Offsets (Version 1),2025-10-28
11,NISAR_L2_GSLC_BETA_V1,NISAR Beta Geocoded Single Look Complex Product (Version 1),2025-10-01
1,NISAR_L2_GSLC_PROVISIONAL_V1,NISAR Provisional Geocoded Single Look Complex Product (Version 1),2025-10-28
24,NISAR_L2_GSLC_V1,NISAR Validated Geocoded Single Look Complex Product (Version 1),2025-10-28
12,NISAR_L2_GUNW_BETA_V1,NISAR Beta Geocoded Unwrapped Interferogram Product (Version 1),2025-10-01


- Each product comes in up to three quality stages: **Beta** (early, not for science), **Provisional** (calibrated, partly validated) and **Validated** (fully validated, only from Oct 21, 2026 on) ([available data](https://nisar-docs.asf.alaska.edu/availability-overview))

- The event we're looking at is in August 2026, so **Provisional** is the one to use

<br/>

The products most likely to be useful for this case:

- **GCOV**: radar backscatter (how bright the ground looks to the radar) for each pass

- **GUNW**: an interferogram comparing two passes 12 days apart, which holds the **coherence**

- **SME2**: soil moisture

([NISAR data products](https://nisar-docs.asf.alaska.edu/products-overview), [GUNW](https://nisar-docs.asf.alaska.edu/gunw))

<br/>

#### 1.2 Granules over the study area

A **granule** is one file: one pass over one area for GCOV and SME2, or one pair of passes for GUNW


The search below asks CMR for every granule of a collection that touches our box within a date range

- `bounding_box`: `west,south,east,north` in degrees

- `temporal`: start and end date

- `.umm_json`: returns the full record for each granule, including the download links

([granule search](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html#granule-search-by-parameters))

<br/>

Study area: the Bhote Koshi valley, from the Langtang Lirung glaciers down to Syabrubesi, drawn on [bboxfinder](http://bboxfinder.com/#28.10,85.25,28.40,85.60). Window: June to October 2026, around the event on Aug 25 to 26

In [20]:
import plotly.graph_objects as go

AOI = (85.25, 28.10, 85.60, 28.40)   # Area of Interest: west, south, east, north
west, south, east, north = AOI

# satellite basemap with the study area drawn on top
fig = go.Figure(go.Scattermap(lon=[west, east, east, west, west], lat=[south, south, north, north, south],
                              mode="lines", line=dict(color="red", width=2), name="study area"))
fig.update_layout(
    map=dict(style="white-bg", center=dict(lon=(west + east) / 2, lat=(south + north) / 2), zoom=9.5,
             layers=[dict(below="traces", sourcetype="raster", sourceattribution="Esri World Imagery",
                          source=["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"])]),
    height=500, margin=dict(l=0, r=0, t=0, b=0), showlegend=False)
fig.show()

In [17]:
START, END = "2026-06-01", "2026-10-31"

# function to find every NISAR file over the study area within the date window
def search_granules(short_name, bbox=AOI, start=START, end=END):
    """
    Search CMR for every granule of one collection over a box and date range

    Args:
        short_name: Collection short name, e.g. "NISAR_L2_GCOV_PROVISIONAL_V1"
        bbox: Bounding box (west, south, east, north) in degrees
        start: First date to include, "YYYY-MM-DD"
        end: Last date to include, "YYYY-MM-DD"

    Returns:
        DataFrame with one row per granule: name, start and end time, size in GB, and the .h5 download link
    """
    rows, page = [], 1
    while True:
        params = {"short_name": short_name, "bounding_box": ",".join(map(str, bbox)),
                  "temporal": f"{start}T00:00:00Z,{end}T23:59:59Z", "page_size": 200, "page_num": page}
        items = requests.get(f"{CMR}/granules.umm_json", params=params, timeout=120).json()["items"]
        for item in items:
            umm = item["umm"]
            urls = [u["URL"] for u in umm["RelatedUrls"] if u["URL"].startswith("https")]
            rows.append({
                "granule": umm["GranuleUR"],
                "start": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]),
                "end": pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["EndingDateTime"]),
                "size_gb": round(umm["DataGranule"]["ArchiveAndDistributionInformation"][0]["SizeInBytes"] / 1e9, 2),
                "url": next(u for u in urls if u.endswith(".h5")),
            })
        # CMR returns at most 200 per page, so a short page means we have them all
        if len(items) < 200:
            break
        page += 1
    return pd.DataFrame(rows)

In [18]:
# search each collection over the study area for June to October
gcov = search_granules("NISAR_L2_GCOV_PROVISIONAL_V1")
gunw = search_granules("NISAR_L2_GUNW_PROVISIONAL_V1")
sme2 = search_granules("NISAR_L3_SME2_PROVISIONAL_V1")
urgent = search_granules("NISAR_UR_L2")

# how many files each one has, and how much it would be to download them all
pd.DataFrame({"granules": [len(gcov), len(gunw), len(sme2), len(urgent)],
              "total_gb": [d.size_gb.sum().round(1) for d in (gcov, gunw, sme2, urgent)]},
             index=["GCOV", "GUNW", "SME2", "Urgent Response"])

,granules,total_gb
GCOV,36,266.0
GUNW,33,77.7
SME2,28,3.4
Urgent Response,30,233.9


- About 580 GB in total; downloading all of that isn't practical, so the plan is to read only the window over our box from each file, remotely

<br/>

#### 1.3 Accessing the files

**Earthdata login**

The `.h5` files need an Earthdata Login token, saved in `dataScience/.env` as `EDL_TOKEN=...` ([how to get a token](https://urs.earthdata.nasa.gov/documentation/for_users/user_token))

In [21]:
from pathlib import Path

# read EDL_TOKEN from dataScience/.env
env = dict(line.split("=", 1) for line in Path("../.env").read_text().splitlines() if "=" in line and not line.startswith("#"))
AUTH = {"Authorization": f"Bearer {env['EDL_TOKEN'].strip()}"}

Check the token works by asking for just the first 8 bytes of one real file. Every HDF5 file starts with the same signature, `\x89HDF\r\n\x1a\n`

- `206` + that signature: the token works

- `200` with a web page: the request got sent to the login page, so the token is missing or expired

In [ ]:
# check if the first GCOV file is accessible with the Earthdata login token
url = gcov.url.iloc[0]
check = requests.get(url, headers={**AUTH, "Range": "bytes=0-7"}, timeout=60)
check.status_code, check.content[:8]

(206, b'\x89HDF\r\n\x1a\n')

<br/>

**Retrieving a file**

The guide mentions files can be streamed instead of downloaded ([Accessing NISAR data](https://nisar-docs.asf.alaska.edu/access-overview)). The download link redirects to a temporary signed link to the file in the cloud, and `h5py` can open it as a file-like object through `fsspec` ([h5py](https://docs.h5py.org/en/stable/high/file.html#python-file-like-objects)), so only the parts we actually read come over the network

In [26]:
import h5py
import fsspec

# function to open a NISAR file in the cloud without downloading the whole thing
def open_h5(url):
    """
    Open a NISAR .h5 file remotely, reading only the parts that get used

    Args:
        url: The .h5 download link from search_granules

    Returns:
        h5py File that can be read like a local file
    """
    # the download link redirects to a temporary signed link, grab it instead of following it
    signed = requests.get(url, headers=AUTH, allow_redirects=False, timeout=60).headers["Location"]
    
    # fsspec fetches the file in 4 MB pieces, only when h5py asks for them
    return h5py.File(fsspec.open(signed, "rb", block_size=4 * 2**20).open(), "r")

In [29]:
# quick test: list the top-level groups of one GCOV file
with open_h5(gcov.url.iloc[0]) as f:
    print(list(f.keys()))
    print(list(f["science/LSAR"].keys()))

['science']
['GCOV', 'identification']


We can now find every NISAR file over the valley and read them remotely. Next is section 2: what's inside a file and how consistent the passes are

<br/>
<br/>

### 2. Exploratory Data Analysis